# Lambert Liu Runner

In [ ]:
import utils as ut
import b_run_staging as b
import h_ll_runner as h
from i_hyper_tuning import Tuner
import c_clustering as c
import numpy as np

### Loading in required data and changing to named tuples

In [ ]:
# Loading in required numpy arrays
static_configs = ut.load_json5("static_configs")
runtime_configs = ut.load_json5("runtime_configs")
config_dict = ut.merge_configs(static_configs, runtime_configs)

hyperparams = ut.load_json5("hyper_choices")
train_test_dict = ut.load_json5("train_test_dict")
bin_metric_dict = ut.load_json5("bin_metric_dict")

user_counts = ut.load_data("user_counts", "df")
user_interactions = ut.load_data("user_interactions", "df")
user_mapping = ut.load_data("user_mapping", "df")

degen_mask = ut.load_data("degen_mask", "np")
interpolation_weights = ut.load_data("interpolation_weights", "np")

# Loading initial grids
u_init = ut.load_data("u_init", "np")
v_init = ut.load_data("v_init", "np")

p_init = ut.load_data("p_init", "np")
u_pos_init = ut.load_data("u_pos_init", "np")
v_pos_init = ut.load_data("v_pos_init", "np")

n_counts_init = ut.load_data("n_counts_init", "np")

u_clustering = ut.load_data("u_clustering", "np")
v_clustering = ut.load_data("v_clustering", "np")

u_pos_clustering = ut.load_data("u_pos_clustering", "np")
v_pos_clustering = ut.load_data("v_pos_clustering", "np")
p_pos_clustering = ut.load_data("p_pos_clustering", "np")


### Converting to named tuples

In [ ]:
# Converting the dfs to nt of numpy arrays to be used for the final numba runner
user_interactions_nt = b.df_to_nt('user_interactions_nt', user_interactions)
user_counts_nt = b.df_to_nt('user_counts_nt', user_counts)
output_idx_nt, model_idx_nt = b.get_model_and_output_idx_nt()
config_nt_class, config_nt, train_test_nt_class, train_test_nt, bin_metric_nt = b.converting_dicts_to_nt(config_dict, train_test_dict, bin_metric_dict)

### Creating tuner class for runs

In [ ]:
if config_dict["hurdle_model"]:
    u_run, v_run, p_run = u_pos_init, v_pos_init, p_init
    u_cluster, v_cluster, p_cluster = u_pos_clustering, v_pos_clustering, p_pos_clustering
else:
    u_run, v_run, p_run = u_init, v_init, np.zeros_like(u_init)
    u_cluster, v_cluster, p_cluster = u_clustering, v_clustering, np.zeros_like(u_clustering)

t = Tuner(u_init, v_init, p_init, u_pos_init, v_pos_init, 
                u_clustering, v_clustering, u_pos_clustering, v_pos_clustering, p_pos_clustering, n_counts_init, 
                user_counts_nt, user_interactions_nt, interpolation_weights, bin_metric_nt, output_idx_nt, model_idx_nt, 
                config_nt_class, train_test_nt_class)

#### Single config runner

In [ ]:
# TODO check we are storing clustering metrics as required
clustering_model = c.make_cluster_model(cluster_param=config_dict["cluster_param"], runtime_configs=config_dict, u_init=u_cluster, v_init=v_cluster, p_init=p_cluster)
output_metrics, calibration_outputs, *_ = t.run_lambert_liu(clustering_model, config_nt=config_nt, train_test_nt=train_test_nt, config_dict=config_dict, degen_mask=degen_mask)

### Multi Config Runner

In [ ]:
results = t.tune_models(hyperparams=hyperparams, train_test_dict=train_test_dict, config_dict=config_dict, degen_mask=degen_mask)